In [ ]:
"""
Zero-shot 3종 비교 ── EXAONE 3.5 7.8B / Qwen3-8B / Kanana 1.5 8B
- valid_all.jsonl(gold 주입된 참고조항 포함)로 순수 판정력 비교
- 각 모델 자기 chat template 사용(공정), 4bit 로드, 하나씩 로드→평가→VRAM 해제
- 출력: 모델별 판정 accuracy / macro-F1 / 포맷준수율 / task별 + 최종 비교표
"""

In [5]:
!pip install -q -U transformers bitsandbytes accelerate scikit-learn hf_transfer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 123.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 106.2 MB/s eta 0:00:00


In [5]:
from google.colab import drive; drive.mount('/content/drive')
VALID = "/content/drive/Othercomputers/내 노트북/project/Workit/data/sft/valid_all.jsonl"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os, re, json, gc, shutil, torch
os.environ["HF_HUB_DISABLE_XET"]="1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"]="1"
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, f1_score
import transformers; print("transformers:", transformers.__version__)   # 5.x 확인

VALID = "valid_all.jsonl"
rows = [json.loads(l) for l in open(VALID, encoding="utf-8")]

MODELS = {
    "exaone-3.5-7.8b": "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct",
    "qwen3-8b":        "Qwen/Qwen3-8B",
    "kanana-1.5-8b":   "kakaocorp/kanana-1.5-8b-instruct-2505",
}

def infer_task(s, g):
    if g in ("일치","불일치","판단보류"): return "contract"
    return "report" if ("결과보고서" in (s or "") or "이행" in (s or "")) else "deliver"

FORMAT = {
    "contract": ("\n\n반드시 아래 형식으로만 답하라:\n판정: <일치|불일치|판단보류>\n"
                 "방향: <을불리|을유리|->\n유형: <A|B|->\n근거: <조항명>\n코멘트: <간단히>"),
    "deliver":  "\n\n반드시 아래 형식으로만 답하라:\n판정: <충족|미흡|불가>\n코멘트: <간단히>",
    "report":   "\n\n반드시 아래 형식으로만 답하라:\n판정: <충족|미흡|불가>\n코멘트: <간단히>",
}

def pj(t):
    m = re.search(r"판정\s*:\s*(일치|불일치|판단보류|충족|미흡|불가)", t)
    return m.group(1) if m else None

def clear_cache(model_id):
    d = os.path.expanduser(f"~/.cache/huggingface/hub/models--{model_id.replace('/','--')}")
    shutil.rmtree(d, ignore_errors=True)
    print(f"  🗑️ 캐시 삭제: {model_id}")

def eval_model(name, model_id):
    print(f"\n[{name}] 로딩...")
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb,
            device_map="auto", trust_remote_code=True).eval()

    yt, yp = [], []
    for r in rows:
        msgs = [dict(m) for m in r["messages"]]
        s = next((m["content"] for m in msgs if m["role"]=="system"), "")
        g = pj(msgs[-1]["content"]); task = infer_task(s, g)
        for m in msgs:
            if m["role"]=="system": m["content"] += FORMAT[task]
        ct = {"add_generation_prompt": True}
        if "qwen3" in name: ct["enable_thinking"] = False        # Qwen thinking off
        ids = tok.apply_chat_template(msgs[:-1], **ct, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=256, do_sample=False, pad_token_id=tok.eos_token_id)
        gen = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
        gen = re.sub(r"<think>.*?</think>", "", gen, flags=re.DOTALL)   # think 제거
        yp.append((pj(gen), task)); yt.append(g)

    pred = [p[0] or "PARSE_FAIL" for p in yp]
    acc = accuracy_score(yt, pred)
    mf1 = f1_score(yt, pred, average="macro", zero_division=0)
    fmt = sum(p[0] is not None for p in yp)/len(yp)
    by = {}
    for t,p,g in zip([x[1] for x in yp], pred, yt): by.setdefault(t,[]).append(p==g)
    task_acc = {t: round(sum(v)/len(v),3) for t,v in by.items()}
    del model, tok; gc.collect(); torch.cuda.empty_cache()
    return {"accuracy":acc, "macro_f1":mf1, "format":fmt, "task_acc":task_acc}

results = {}
for name, mid in MODELS.items():
    try:
        results[name] = eval_model(name, mid)
        r = results[name]
        print(f"[{name}] acc={r['accuracy']:.3f} macroF1={r['macro_f1']:.3f} "
              f"포맷={r['format']:.3f} task={r['task_acc']}")
    except Exception as e:
        print(f"[{name}] ❌ 실패: {type(e).__name__}: {e}")
        results[name] = None
    finally:
        clear_cache(mid)                      # 매 모델 끝나면 캐시 삭제
        gc.collect(); torch.cuda.empty_cache()

print(f"\n{'='*55}\n  Zero-shot 비교 (n={len(rows)})\n{'='*55}")
print(f"{'모델':<18}{'정확도':>8}{'macroF1':>10}{'포맷':>8}")
for name, r in results.items():
    print(f"{name:<18}{r['accuracy']:>8.3f}{r['macro_f1']:>10.3f}{r['format']:>8.3f}" if r
          else f"{name:<18}{'로드실패':>8}")
print("\ntask별 정확도:")
for name, r in results.items():
    if r: print(f"  {name}: {r['task_acc']}")

transformers: 5.12.1

[exaone-3.5-7.8b] 로딩...


[transformers] The `check_model_inputs` decorator is deprecated in favor of `merge_with_config_defaults`.


[ERROR] `cache_position` is part of ExaoneModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in /root/.cache/huggingface/modules/transformers_modules/LGAI_hyphen_EXAONE/EXAONE_hyphen_3_dot_5_hyphen_7_dot_8B_hyphen_Instruct/553ea250b9a5317231459279d5847d6cf955b9aa/modeling_exaone.py.
[ERROR] `cache_position` is part of ExaoneForCausalLM.forward's signature, but not documented. Make sure to add it to the docstring of the function in /root/.cache/huggingface/modules/transformers_modules/LGAI_hyphen_EXAONE/EXAONE_hyphen_3_dot_5_hyphen_7_dot_8B_hyphen_Instruct/553ea250b9a5317231459279d5847d6cf955b9aa/modeling_exaone.py.


model.safetensors.index.json:   0%|          | 0.00/23.7k [00:00<?, ?B/s]

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


KeyboardInterrupt: 

In [10]:
!df -h /
!du -sh ~/.cache/huggingface 2>/dev/null
!du -sh /content/* 2>/dev/null | sort -h | tail

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   77G   36G  69% /
188K	/root/.cache/huggingface
^C
